In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


def find_project_root(start_path: Path, required_dir: str = "src") -> Path:
    current = start_path.resolve()
    while True:
        if (current / required_dir).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Cannot find project root containing '{required_dir}'")
        current = current.parent


PROJECT_ROOT = find_project_root(Path.cwd(), required_dir="src")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import (
    MultiModalRawDataset,
    multimodal_raw_collate_fn,
)
from src.models.multimodal.pipeline.pipeline_text_guided_pvd import (
    MultiModalPipelineTextGuidedPVD,
)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /media/data3/users/luongdth/MulCo-PlantNet


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [3]:
# =========================
# RAW MULTIMODAL TEST SPLIT
# =========================
VAL_IMAGE_ROOT = PROJECT_ROOT / "data" / "AIDG" / "dataset_PlantDoc" / "images" / "test"
VAL_CAPTION_ROOT = PROJECT_ROOT / "data" / "AIDG" / "captions_LLaVA" / "test"

# File tốt của ver2 là state_dict
CKPT_PATH = PROJECT_ROOT / "archive" / "multimodal_text_guided" / "multimodal_text_guided_pvd_classifier_state_dict.pth"

NUM_CLASSES = 28
BATCH_SIZE = 8
IMG_SIZE = 224
NUM_WORKERS = 0

SAVE_DIR = PROJECT_ROOT / "archive" / "multimodal_text_guided" / "test_results_raw"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = SAVE_DIR / "classification_report.csv"
CM_PATH = SAVE_DIR / "confusion_matrix.csv"
PRED_PATH = SAVE_DIR / "predictions.csv"

print("VAL_IMAGE_ROOT:", VAL_IMAGE_ROOT)
print("VAL_CAPTION_ROOT:", VAL_CAPTION_ROOT)
print("CKPT_PATH:", CKPT_PATH)
print("SAVE_DIR:", SAVE_DIR)

VAL_IMAGE_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/dataset_PlantDoc/images/test
VAL_CAPTION_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/captions_LLaVA/test
CKPT_PATH: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal_text_guided/multimodal_text_guided_pvd_classifier_state_dict.pth
SAVE_DIR: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal_text_guided/test_results_raw


In [4]:
print("VAL_IMAGE_ROOT exists:", VAL_IMAGE_ROOT.exists())
print("VAL_CAPTION_ROOT exists:", VAL_CAPTION_ROOT.exists())
print("CKPT_PATH exists:", CKPT_PATH.exists())

if not VAL_IMAGE_ROOT.exists():
    raise FileNotFoundError(f"Validation image root not found: {VAL_IMAGE_ROOT}")
if not VAL_CAPTION_ROOT.exists():
    raise FileNotFoundError(f"Validation caption root not found: {VAL_CAPTION_ROOT}")
if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

VAL_IMAGE_ROOT exists: True
VAL_CAPTION_ROOT exists: True
CKPT_PATH exists: True


In [5]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [6]:
dataset = MultiModalRawDataset(
    image_root=VAL_IMAGE_ROOT,
    caption_root=VAL_CAPTION_ROOT,
    transform=transform,
    use_depth_suppressed=False,
    strict_caption_match=True,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=multimodal_raw_collate_fn,
    pin_memory=(device == "cuda"),
)

print("Validation samples:", len(dataset))
print("Num classes:", len(dataset.class_to_idx))
print("class_to_idx:", dataset.class_to_idx)

[MultiModalRawDataset] Total selected images: 236
[MultiModalRawDataset] Valid samples: 236
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Num classes: 27
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'grape_leaf': 25, 'grape_leaf_black_rot': 26}
Validation samples: 236
Num classes: 27
class_

In [7]:
def main():
    run_device = device

    print("Using device:", run_device)
    print("VAL_IMAGE_ROOT:", VAL_IMAGE_ROOT)
    print("VAL_CAPTION_ROOT:", VAL_CAPTION_ROOT)
    print("CKPT_PATH:", CKPT_PATH)

    # Tự fallback về CPU nếu CUDA OOM lúc load model
    try:
        model = MultiModalPipelineTextGuidedPVD(
            ckpt_path=CKPT_PATH,
            num_classes=NUM_CLASSES,
            clip_model_name="ViT-L-14",
            clip_pretrained="openai",
            text_input_dim=768,
            image_input_dim=1024,
            proj_dim=512,
            proj_hidden_dim=768,
            pvd_hidden_dim=768,
            cls_hidden_dim=1024,
            dropout=0.3,
            normalize_projection=False,
            device=run_device,
        ).to(run_device)
    except torch.OutOfMemoryError:
        print("[WARN] CUDA OOM while loading model. Falling back to CPU...")
        run_device = "cpu"
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = MultiModalPipelineTextGuidedPVD(
            ckpt_path=CKPT_PATH,
            num_classes=NUM_CLASSES,
            clip_model_name="ViT-L-14",
            clip_pretrained="openai",
            text_input_dim=768,
            image_input_dim=1024,
            proj_dim=512,
            proj_hidden_dim=768,
            pvd_hidden_dim=768,
            cls_hidden_dim=1024,
            dropout=0.3,
            normalize_projection=False,
            device=run_device,
        ).to(run_device)

    model.eval()

    y_true = []
    y_pred = []
    prediction_rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating raw multimodal (ver2)"):
            images = batch["image"].to(run_device, non_blocking=True)
            texts = batch["text"]
            labels = batch["label"].to(run_device, non_blocking=True)

            logits = model(images, texts)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            for i in range(len(texts)):
                true_idx = labels[i].item()
                pred_idx = preds[i].item()

                prediction_rows.append({
                    "image_name": batch["image_name"][i],
                    "image_path": batch["image_path"][i],
                    "text": texts[i],
                    "true_label_idx": true_idx,
                    "pred_label_idx": pred_idx,
                    "true_class_name": batch["class_name"][i],
                    "pred_class_name": dataset.idx_to_class[pred_idx],
                    "correct": int(true_idx == pred_idx),
                    "source_json": batch["source_json"][i],
                })

    acc = accuracy_score(y_true, y_pred)
    print(f"Validation Accuracy: {acc:.4f}")

    class_names = [dataset.idx_to_class[i] for i in range(len(dataset.idx_to_class))]

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred)

    report_df = pd.DataFrame(report).transpose()
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    pred_df = pd.DataFrame(prediction_rows)

    report_df.to_csv(REPORT_PATH, encoding="utf-8-sig")
    cm_df.to_csv(CM_PATH, encoding="utf-8-sig")
    pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")

    print("Saved classification report to:", REPORT_PATH)
    print("Saved confusion matrix to:", CM_PATH)
    print("Saved predictions to:", PRED_PATH)

In [8]:
main()

Using device: cuda
VAL_IMAGE_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/dataset_PlantDoc/images/test
VAL_CAPTION_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/captions_LLaVA/test
CKPT_PATH: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal_text_guided/multimodal_text_guided_pvd_classifier_state_dict.pth


/media/data3/users/luongdth/anaconda3/envs/gr1/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Validating raw multimodal (ver2):   0%|          | 0/30 [00:00<?, ?it/s]

Validation Accuracy: 0.7203
Saved classification report to: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal_text_guided/test_results_raw/classification_report.csv
Saved confusion matrix to: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal_text_guided/test_results_raw/confusion_matrix.csv
Saved predictions to: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal_text_guided/test_results_raw/predictions.csv
